In [ ]:
import datasets
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSequenceClassification
import torch

In [ ]:
id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2, id2label=id2label, label2id=label2id, dtype=torch.float32)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# dataset = datasets.load_dataset("stanfordnlp/sst2")
dataset = datasets.load_dataset("fancyzhx/amazon_polarity")

README.md:   0%|          | 0.00/6.81k [00:00<?, ?B/s]

amazon_polarity/train-00000-of-00004.par(…): reconstructing file:   0%|          |  0.00B /  260MB            

amazon_polarity/train-00000-of-00004.par(…): downloading bytes:           |  0.00B            

amazon_polarity/train-00001-of-00004.par(…): reconstructing file:   0%|          |  0.00B /  258MB            

amazon_polarity/train-00001-of-00004.par(…): downloading bytes:           |  0.00B            

...

amazon_polarity/train-00003-of-00004.par(…): downloading bytes:           |  0.00B            

amazon_polarity/test-00000-of-00001.parq(…): reconstructing file:   0%|          |  0.00B /  117MB            

amazon_polarity/test-00000-of-00001.parq(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3600000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400000 [00:00<?, ? examples/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 3600000
    })
    test: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 400000
    })
})

Load tokenizer, tokenize dataset

In [ ]:
# import sacremoses

tok = AutoTokenizer.from_pretrained(MODEL)
# END = "\n"
# tok.pad_token = tok.eos_token
# sacremoses.MosesDetokenizer().detokenize(s.split())

# take a smaller sample
subsample = dataset["train"].train_test_split(
    train_size=6000, stratify_by_column="label", seed=0)["train"]

split = subsample.train_test_split(
    train_size=5000, stratify_by_column="label", seed=0)
tokenized_dataset = split
tokenized_dataset["train"] = split["train"].map(lambda x: tok(x["text"], truncation=True), batched=True)
tokenized_dataset["validation"] = split["test"].map(lambda x: tok(x["text"], truncation=True), batched=True)
tokenized_dataset["test"] = dataset["test"].map(lambda x: tok(x["text"], truncation=True), batched=True)
# padding needed for minibatch training
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tok)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 50000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 38000
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 10000
    })
})

In [ ]:
model.base_model.requires_grad_(False)
model.score.requires_grad_(True)
model.eval()   # score has no dropout, so eval on the whole model is fine

Qwen2ForSequenceClassification(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rota

function to compute metrics

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

Training loop for the classifier head

In [ ]:
from torch.utils.data import DataLoader

device = "cpu"
model.to(device)

cols = ["input_ids", "attention_mask", "label"]  # collator renames label -> labels
train_dl = DataLoader(tokenized_dataset["train"].select_columns(cols), batch_size=1, shuffle=True, collate_fn=data_collator)
val_dl = DataLoader(tokenized_dataset["validation"].select_columns(cols), batch_size=1, collate_fn=data_collator)

For efficiency, pre-calculate features produced by the transformer; train the classifier on those features

In [ ]:
from tqdm import tqdm

def extract_llm_features(dataloader):
    per_sentence_transformer_feats = []
    labels = []
    with torch.inference_mode():
        for batch in tqdm(dataloader):
            batch = {k: v.to(device) for k, v in batch.items()}
            # shape [B, L, 768]
            batch_last_hidden_states = model.base_model(**batch).last_hidden_state
            # find the index of the last non-padding token so we get the state after it
            batch_last_nonpadding_token_idxs = batch['attention_mask'].sum(1) - 1
            # shape [L, 768]
            for per_sentence_last_hidden_states, last_nonpadding_token_idx in zip(batch_last_hidden_states, batch_last_nonpadding_token_idxs):
                per_sentence_transformer_feats.append(
                    per_sentence_last_hidden_states[last_nonpadding_token_idx]
                )
            labels.append(batch['labels'])
    return per_sentence_transformer_feats, labels

scorer_train_feats, scorer_train_labels = extract_llm_features(train_dl)
scorer_val_feats, scorer_val_labels = extract_llm_features(val_dl)

scorer_train_feats = torch.vstack(scorer_train_feats)
scorer_train_labels = torch.concat(scorer_train_labels)

scorer_val_feats = torch.vstack(scorer_val_feats)
scorer_val_labels = torch.concat(scorer_val_labels)

torch.save(scorer_train_feats, f"./saved_features/qwen25_05b_feat_train.pt")
torch.save(scorer_val_feats, f"./saved_features/qwen25_05b_feat_val.pt")
torch.save(scorer_train_labels, f"./saved_features/qwen25_05b_labels_train.pt")
torch.save(scorer_val_labels, f"./saved_features/qwen25_05b_labels_val.pt")

  0%|          | 14/50000 [00:04<4:07:22,  3.37it/s]


KeyboardInterrupt: 

In [ ]:
scorer_train_feats = torch.load(f"./saved_features/yelp/qwen25_05b_feat_train.pt")
scorer_val_feats = torch.load(f"./saved_features/yelp/qwen25_05b_feat_val.pt")
scorer_train_labels = torch.load(f"./saved_features/yelp/qwen25_05b_labels_train.pt")
scorer_val_labels = torch.load(f"./saved_features/yelp/qwen25_05b_labels_val.pt")

In [ ]:
from torch.utils.data import TensorDataset

ds = TensorDataset(scorer_train_feats, scorer_train_labels)
dl = DataLoader(ds, batch_size=128, shuffle=True)
loss_fn = torch.nn.CrossEntropyLoss()
opt = torch.optim.AdamW(model.score.parameters(), lr=1e-3, weight_decay=0.1)
best_val_loss = 1
patience = 50
epochs_val_plateau = 0

for epoch in range(1000):
    with torch.inference_mode():
        val_logits = model.score(scorer_val_feats)
        val_loss = loss_fn(val_logits, scorer_val_labels)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_val_plateau = 0
        else:
            epochs_val_plateau +=1
            if epochs_val_plateau == patience:
                break

        if epoch % 10 == 0:
            print(
                compute_metrics((val_logits.detach().numpy(), scorer_val_labels.detach().numpy())), val_loss.detach().numpy()
            )

    for b_features, b_labels in dl:
        out = model.score(b_features)
        loss = loss_fn(out, b_labels)
        loss.backward()
        opt.step()
        opt.zero_grad()


/home/andre/anaconda3/envs/llm-cuda/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'accuracy': 0.4908256880733945, 'f1': 0.0, 'precision': 0.0, 'recall': 0.0} 2.0991244
{'accuracy': 0.9025229357798165, 'f1': 0.9008168028004667, 'precision': 0.9346246973365617, 'recall': 0.8693693693693694} 0.2526524
{'accuracy': 0.8956422018348624, 'f1': 0.9014084507042254, 'precision': 0.8684759916492694, 'recall': 0.9369369369369369} 0.25805
{'accuracy': 0.8910550458715596, 'f1': 0.8986125933831377, 'precision': 0.8539553752535497, 'recall': 0.9481981981981982} 0.27369583
{'accuracy': 0.9013761467889908, 'f1': 0.9006928406466512, 'precision': 0.9241706161137441, 'recall': 0.8783783783783784} 0.2472523
{'accuracy': 0.9002293577981652, 'f1': 0.9012485811577753, 'precision': 0.9084668192219679, 'recall': 0.8941441441441441} 0.24542083
{'accuracy': 0.8922018348623854, 'f1': 0.8980477223427332, 'precision': 0.8661087866108786, 'recall': 0.9324324324324325} 0.28385893
{'accuracy': 0.8967889908256881, 'f1': 0.9006622516556292, 'precision': 0.8831168831168831, 'recall': 0.918918918918919}

In [ ]:
# score GPT-2 generations as in the loop: one-hot tokens -> Y @ E_J -> head reads the last position
gen_model = AutoModelForCausalLM.from_pretrained(MODEL).to(device).eval()
# E_J = model.transformer.wte.weight
E_J = model.get_input_embeddings().weight
SUFFIX = tok(" ")["input_ids"]  # SST-2 sentences end with a trailing space, so the head was fit on that token's state
BASE = "Write one short sentence about trains."
INSTRUCTIONS = [
    "Be extremely enthusiastic and positive.",
    "Be strictly neutral, impersonal, unemotional.",
    "Be extremely critical and negative.",
]

PROMPTS = [BASE + i for i in INSTRUCTIONS]

with torch.inference_mode():
    for prompt in PROMPTS:
        prompt_ids = tok.apply_chat_template(
            [{"role": "user", "content": f"{prompt}"}],
             add_generation_prompt=True,
             return_tensors="pt"
            )["input_ids"].to(device)
        
        for seed in range(3):
            torch.manual_seed(seed)
            ids = gen_model.generate(prompt_ids, do_sample=True, max_new_tokens=100, pad_token_id=tok.eos_token_id)[0, prompt_ids.shape[1]:]
            ids = ids[ids != tok.eos_token_id]
            Y = torch.nn.functional.one_hot(ids, E_J.shape[0]).float()
            emb = torch.cat([Y @ E_J, E_J[SUFFIX]])
            logits = model(inputs_embeds=emb[None]).logits[0]
            s = logits[1] - logits[0]  # log-odds of positive
            print(f"{s.item():+.2f}  {prompt}|{tok.decode(ids)!r}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

+18.74  Write one short sentence about trains.Be extremely enthusiastic and positive.|'Imagine a world where every day is filled with the rhythmic beat of speeding trains carrying joyous passengers across vibrant landscapes!'
+11.88  Write one short sentence about trains.Be extremely enthusiastic and positive.|'Trains zoom through the vast landscapes, transporting millions of people across the world every day with unparalleled speed and efficiency!'
+18.77  Write one short sentence about trains.Be extremely enthusiastic and positive.|'Imagine a world where every morning you wake up to the sound of trilling engines as sleek, futuristic trains glide through the sky, connecting cities with unparalleled speed and efficiency!'
+15.83  Write one short sentence about trains.Be strictly neutral, impersonal, unemotional.|'Trains efficiently transport passengers across vast distances with minimal disruption to schedules.'
+17.82  Write one short sentence about trains.Be strictly neutral, imperso